# M21 lab solution: Framework Fluency: LlamaIndex

**Brief.** Port one of your Part V pipelines (hybrid + rerank) to LlamaIndex and confirm the metrics match.

Run with the `llamaindex` and `hf` groups: `uv run --group llamaindex --group hf jupyter lab` (LlamaIndex, plus sentence-transformers for the cross-encoder).

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # API keys from the .env file at the repo root
sys.path.insert(0, str(Path("../../../../src").resolve()))

from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k  # noqa: E402

DATA = Path("../data/larkspur")
handbook = load_handbook(DATA / "handbook")
golden = load_jsonl(DATA / "golden.jsonl")
by_id = {g["id"]: g for g in golden}
scored = [g for g in golden if g["evidence_spans"]]
print(len(handbook), "handbook files,", len(golden), "golden questions,", len(scored), "with evidence")

In [ ]:
from helpers import get_embeddings, get_experientiallabs_llm

llm = get_experientiallabs_llm()
embeddings = get_embeddings()
LLM_NAME = llm.model_name
EMBEDDING_NAME = getattr(embeddings, "model", None) or getattr(embeddings, "endpoint", None)
print("LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)

In [ ]:
def texts(docs):
    return [d.page_content for d in docs]


def score(results, questions=None):
    """results: question id -> retrieved chunk texts in rank order; scored with lab_kit on the questions with evidence."""
    qs = questions or golden
    return {"hit_rate": round(hit_rate(results, qs), 3), "recall_at_k": round(recall_at_k(results, qs), 3),
            "mrr": round(mrr(results, qs), 3)}

## 1. The same chunks on both sides

Both pipelines index the identical 30 chunks (the `lab_kit.baseline` split) and embed them with the same model, so a difference in the metrics comes from retrieval, fusion or reranking, not from the data.

The pipeline: BM25 and dense retrieval (top 20 each), Reciprocal Rank Fusion, a `cross-encoder/ms-marco-MiniLM-L-6-v2` rerank, top 4.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
pieces = [(c, {"source": p["source"], "section": p["title"]}) for p in handbook for c in splitter.split_text(p["text"])]
DEPTH, K = 20, 4
print(len(pieces), "chunks")

## 2. LangChain: EnsembleRetriever and a cross-encoder

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from sentence_transformers import CrossEncoder

lc_docs = [Document(page_content=t, metadata=m) for t, m in pieces]
lc_hybrid = EnsembleRetriever(
    retrievers=[BM25Retriever.from_documents(lc_docs, k=DEPTH),
                FAISS.from_documents(lc_docs, embeddings).as_retriever(search_kwargs={"k": DEPTH})],
    weights=[0.5, 0.5],
)
cross = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def lc_pipeline(question):
    cands = [d.page_content for d in lc_hybrid.invoke(question)][:DEPTH]
    scores = cross.predict([(question, c) for c in cands])
    return [c for _, c in sorted(zip(scores, cands), key=lambda x: -x[0])][:K]


lc_results = {g["id"]: lc_pipeline(g["question"]) for g in golden}
print("LangChain:", score(lc_results))

## 3. LlamaIndex: the port

- Nodes are `TextNode`s with the same text and metadata. The metadata is excluded from the embedded and LLM text, because LlamaIndex otherwise prepends it, and the two sides would embed different strings.
- The embedding model is the same LangChain object, wrapped as a LlamaIndex `BaseEmbedding`.
- `QueryFusionRetriever` fuses the vector and BM25 retrievers with `mode="reciprocal_rerank"`. `num_queries=1` keeps the original question only (the default writes extra queries with an LLM), so a `MockLLM` is enough.
- `SentenceTransformerRerank` runs the same cross-encoder and keeps the top 4.

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.embeddings import BaseEmbedding
from llama_index.core.llms import MockLLM
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.schema import TextNode
from llama_index.retrievers.bm25 import BM25Retriever as LIBM25Retriever


class LangChainEmbedding(BaseEmbedding):
    """The LangChain embeddings above, so both pipelines embed with the same model."""

    def _get_query_embedding(self, query):
        return embeddings.embed_query(query)

    def _get_text_embedding(self, text):
        return embeddings.embed_documents([text])[0]

    def _get_text_embeddings(self, texts_):
        return embeddings.embed_documents(texts_)

    async def _aget_query_embedding(self, query):
        return self._get_query_embedding(query)


nodes = [TextNode(text=t, metadata=m, excluded_embed_metadata_keys=list(m), excluded_llm_metadata_keys=list(m))
         for t, m in pieces]
li_index = VectorStoreIndex(nodes, embed_model=LangChainEmbedding())
li_hybrid = QueryFusionRetriever(
    [li_index.as_retriever(similarity_top_k=DEPTH), LIBM25Retriever.from_defaults(nodes=nodes, similarity_top_k=DEPTH)],
    similarity_top_k=DEPTH, num_queries=1, mode="reciprocal_rerank", use_async=False, llm=MockLLM(),
)
li_rerank = SentenceTransformerRerank(model="cross-encoder/ms-marco-MiniLM-L-6-v2", top_n=K)


def li_pipeline(question):
    found = li_rerank.postprocess_nodes(li_hybrid.retrieve(question), query_str=question)
    return [n.node.get_content() for n in found]


li_results = {g["id"]: li_pipeline(g["question"]) for g in golden}
print("LlamaIndex:", score(li_results))

## 4. Compare and explain the gap

The table puts the two pipelines side by side. The cells after it list every question where the two disagree on a hit, with the stage-1 candidates of each, so a gap can be traced to BM25 (the two libraries tokenise differently: LangChain's `BM25Retriever` splits on whitespace), to fusion, or to the rerank.

In [ ]:
import pandas as pd

print("run:", time.strftime("%Y-%m-%d"), "| embeddings:", EMBEDDING_NAME, "| reranker: ms-marco-MiniLM-L-6-v2")
print(pd.DataFrame([{"pipeline": "LangChain", **score(lc_results)}, {"pipeline": "LlamaIndex", **score(li_results)}]).to_string(index=False))

diff = []
for g in scored:
    a = any(is_hit(c, g["evidence_spans"]) for c in lc_results[g["id"]])
    b = any(is_hit(c, g["evidence_spans"]) for c in li_results[g["id"]])
    if a != b:
        diff.append(g)
        print(f"{g['id']} LangChain hit={a} LlamaIndex hit={b}: {g['question']}")
print(len(diff), "questions differ")

In [ ]:
for g in diff:
    lc_c = [d.page_content for d in lc_hybrid.invoke(g["question"])][:DEPTH]
    li_c = [n.node.get_content() for n in li_hybrid.retrieve(g["question"])]
    rank = lambda cands: next((i for i, c in enumerate(cands, 1) if is_hit(c, g["evidence_spans"])), None)
    print(g["id"], "| first evidence rank in stage 1: LangChain", rank(lc_c), "LlamaIndex", rank(li_c))

## What this shows

- With the chunks, the embedding model and the reranker held equal, the remaining differences are BM25's tokeniser and scoring, and how each library's fusion breaks ties. The stage-1 ranks above show which stage a disagreement starts in.
- If the evidence is in both stage-1 lists, the rerank sees the same pairs and should agree; a disagreement that starts in stage 1 is a BM25 or fusion difference.
- A port is only "the same pipeline" once its defaults are pinned: LlamaIndex prepends metadata to the embedded text and `QueryFusionRetriever` writes extra queries unless told otherwise. Both were switched off here on purpose.